<a href="https://colab.research.google.com/github/chewanna7-code/NatureInsightStudy/blob/main/Rainfall_Event_Pattern_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Rainfall Event Pattern and Antecedent Analysis

## Purpose

This notebook extracts, plots and summarises rainfall data surrounding a selected flood event and can be compared to other events.

It is designed as a reusable workflow, rather than being specific to one catchment. The user can adapt the catchment name, event date, rainfall dataset and analysis window depending on the study area.

---

## What This Notebook Does

- Uploads a daily rainfall dataset.
- Cleans and standardises date and rainfall columns.
- Extracts rainfall before and after a selected event date.
- Calculates cumulative rainfall across the event window.
- Plots daily rainfall and cumulative rainfall.
- Produces a short summary table for interpretation.
- Saves the extracted event rainfall data and figure.

---

## Where Alterations Are Needed

Edit the settings cell below to adapt the notebook for a different catchment, event or dataset.

The main values to check are:

- `catchment_name`
- `event_name`
- `event_date`
- `days_before`
- `days_after`
- `date_column`
- `rainfall_column`

The column names must match the uploaded rainfall file.

---

## How This Supports NatureInsight® Analysis

NatureInsight® flood hydrographs are based on simplified design rainfall assumptions. Historic flood events, however, are often influenced by rainfall patterns over several days, not only rainfall on the peak-flow day.

This workflow helps assess whether an observed flood event was associated with:

- A short, intense rainfall pulse.
- Several days of antecedent rainfall.
- Successive rainfall peaks.
- A prolonged wet period before the flood peak.

This is useful when comparing NatureInsight® outputs with observed hydrological behaviour, because differences in rainfall pattern can help explain why modelled hydrographs may not fully reflect real catchment response.

---

## How to Interpret Rainfall Shapes

### 1. Single sharp rainfall peak

A single large bar may suggest a short-duration intense rainfall event. This can produce a fast catchment response, especially in steep, urbanised or low-infiltration catchments.

### 2. Multiple rainfall peaks

Several rainfall pulses close together may indicate repeated storm activity. Catchments may become progressively wetter, reducing infiltration capacity and increasing runoff response.

### 3. Gradual cumulative rainfall rise

A steady increase in cumulative rainfall suggests prolonged wetting. This can indicate antecedent saturation, where soils may already be wet before the main flood-producing rainfall occurs.

### 4. Rainfall peak before flow peak

If rainfall peaks before the river flow peak, this may reflect catchment lag time. Larger catchments or flatter catchments may show a slower response.

### 5. Rainfall and flow peak close together

A close rainfall-flow response can suggest flashy catchment behaviour. This may occur where slopes are steep, soils are less permeable, drainage pathways are efficient, or urban surfaces are present.

---

## Outputs

The notebook exports:

- A CSV file containing the rainfall event window.
- A PNG figure showing rainfall pattern and cumulative rainfall.
- A small summary table of rainfall totals.

## Example outputs

![Hydrograph and Rainfall Extraction Example](images/Calder_Example_for_Extraction_Hydrographs_and_Rainfall.png)

![Hydrograph and Rainfall Extraction Example](images/Flood_Frequency_Mitford_Output_Example.png)



## Step 1 – Edit Analysis Settings

Change the values in this cell to match the catchment, event and rainfall dataset being analysed.

This is the main section users need to alter when reusing the notebook.


In [ ]:

# ============================================================
# USER SETTINGS - EDIT THESE VALUES
# ============================================================

# Name of catchment or study area used in plot titles and output files
catchment_name = "Example Catchment"

# Name of flood event or rainfall event
event_name = "Example Flood Event"

# Main event date, usually the observed flood peak date
# Format: YYYY-MM-DD
event_date = "2008-01-21"

# Number of days to include before and after the event date
days_before = 6
days_after = 3

# Column names in the uploaded rainfall dataset
# These must match the uploaded file exactly
date_column = "Date"
rainfall_column = "Daily Rainfall (mm)"

# Optional: set to True if the uploaded file has no header row
file_has_no_header = False

# Optional output name prefix
output_prefix = "rainfall_event_analysis"



## Step 2 – Import Libraries

These libraries are used for:

- File upload in Google Colab.
- Reading rainfall data.
- Cleaning dates and numeric rainfall values.
- Plotting rainfall patterns.
- Exporting results.


In [ ]:

# Import required libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
from pathlib import Path



## Step 3 – Upload Rainfall Dataset

Upload the rainfall file when prompted.

Supported formats:

- `.csv`
- `.xlsx`
- `.xls`
- `.txt`

For text files, the code will attempt to read the data using common separators.


In [ ]:

# Upload rainfall dataset

uploaded = files.upload()

if len(uploaded) == 0:
    raise ValueError("No file uploaded.")

file_name = list(uploaded.keys())[0]
print(f"Uploaded file: {file_name}")



## Step 4 – Load and Check the Rainfall Data

This section reads the uploaded rainfall file and displays the first few rows.

If the notebook raises an error here, check:

- The file type.
- Whether the file has headings.
- Whether the date and rainfall column names match the settings cell.


In [ ]:

# Load rainfall dataset depending on file type

suffix = Path(file_name).suffix.lower()

if suffix in [".xlsx", ".xls"]:
    df = pd.read_excel(file_name, header=None if file_has_no_header else 0)

elif suffix == ".csv":
    df = pd.read_csv(file_name, header=None if file_has_no_header else 0)

elif suffix == ".txt":
    # Tries common separators for text rainfall files
    df = pd.read_csv(file_name, sep=None, engine="python", header=None if file_has_no_header else 0)

else:
    raise ValueError(f"Unsupported file type: {suffix}")

# If file has no header, assume first column is date and second column is rainfall
if file_has_no_header:
    df = df.iloc[:, :2]
    df.columns = [date_column, rainfall_column]

print("Data preview:")
display(df.head())

print("\nAvailable columns:")
print(list(df.columns))



## Step 5 – Clean Date and Rainfall Columns

This section converts the selected columns into a consistent format.

Dates are converted to `datetime` format and rainfall values are converted to numeric values.

Any invalid rainfall entries are set to missing values and removed from the event extraction.


In [ ]:

# Check required columns exist

if date_column not in df.columns:
    raise KeyError(
        f"Date column '{date_column}' was not found. "
        f"Available columns are: {list(df.columns)}"
    )

if rainfall_column not in df.columns:
    raise KeyError(
        f"Rainfall column '{rainfall_column}' was not found. "
        f"Available columns are: {list(df.columns)}"
    )

# Clean and standardise columns
rain = df[[date_column, rainfall_column]].copy()
rain.columns = ["date", "rainfall_mm"]

rain["date"] = pd.to_datetime(rain["date"], errors="coerce", dayfirst=True)
rain["rainfall_mm"] = (
    rain["rainfall_mm"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("mm", "", regex=False)
)
rain["rainfall_mm"] = pd.to_numeric(rain["rainfall_mm"], errors="coerce")

# Remove rows with missing dates or rainfall
rain = rain.dropna(subset=["date", "rainfall_mm"]).sort_values("date").reset_index(drop=True)

print("Cleaned rainfall data:")
display(rain.head())

print(f"\nDate range available: {rain['date'].min().date()} to {rain['date'].max().date()}")



## Step 6 – Extract the Event Window

The event window includes rainfall before and after the selected event date.

This helps identify whether the flood event followed a dry period, a short intense storm, or several days of rainfall accumulation.


In [ ]:

# Define event window

event_dt = pd.to_datetime(event_date)

start_date = event_dt - pd.Timedelta(days=days_before)
end_date = event_dt + pd.Timedelta(days=days_after)

event_rain = rain[
    (rain["date"] >= start_date) &
    (rain["date"] <= end_date)
].copy()

if event_rain.empty:
    raise ValueError(
        "No rainfall data found in the selected event window. "
        "Check event_date, days_before/days_after, and dataset date range."
    )

# Add event-relative fields
event_rain["days_from_event"] = (event_rain["date"] - event_dt).dt.days
event_rain["cumulative_rainfall_mm"] = event_rain["rainfall_mm"].cumsum()

print(f"Event window: {start_date.date()} to {end_date.date()}")
display(event_rain)



## Step 7 – Summarise Rainfall Conditions

This table provides simple rainfall indicators that can be used in discussion.

Useful interpretation points:

- High rainfall on the event day may suggest direct storm response.
- High rainfall before the event may suggest antecedent wetness or saturation.
- High total rainfall across the window may help explain larger observed flood peaks.


In [ ]:

# Calculate rainfall summary metrics

rain_before = event_rain[event_rain["date"] < event_dt]["rainfall_mm"].sum()
rain_on_event = event_rain[event_rain["date"] == event_dt]["rainfall_mm"].sum()
rain_after = event_rain[event_rain["date"] > event_dt]["rainfall_mm"].sum()
total_rain = event_rain["rainfall_mm"].sum()
max_daily_rain = event_rain["rainfall_mm"].max()
max_daily_date = event_rain.loc[event_rain["rainfall_mm"].idxmax(), "date"]

summary = pd.DataFrame({
    "Metric": [
        "Rainfall before event date (mm)",
        "Rainfall on event date (mm)",
        "Rainfall after event date (mm)",
        "Total rainfall in event window (mm)",
        "Maximum daily rainfall (mm)",
        "Date of maximum daily rainfall",
    ],
    "Value": [
        round(rain_before, 2),
        round(rain_on_event, 2),
        round(rain_after, 2),
        round(total_rain, 2),
        round(max_daily_rain, 2),
        max_daily_date.date(),
    ]
})

display(summary)



## Step 8 – Plot Daily and Cumulative Rainfall

The figure shows:

- Daily rainfall as bars.
- Cumulative rainfall as a line.
- The selected event date as a vertical marker.

This allows rainfall pattern and antecedent accumulation to be interpreted visually.


In [ ]:

# Plot rainfall event pattern

plt.rcParams["font.family"] = "serif"

fig, ax1 = plt.subplots(figsize=(10, 5))

# Daily rainfall bars
ax1.bar(event_rain["date"], event_rain["rainfall_mm"], label="Daily rainfall")
ax1.set_xlabel("Date")
ax1.set_ylabel("Daily rainfall (mm)")
ax1.tick_params(axis="x", rotation=45)

# Event date marker
ax1.axvline(event_dt, linestyle="--", linewidth=1.5, label="Event date")

# Cumulative rainfall line on second axis
ax2 = ax1.twinx()
ax2.plot(event_rain["date"], event_rain["cumulative_rainfall_mm"], marker="o", label="Cumulative rainfall")
ax2.set_ylabel("Cumulative rainfall (mm)")

# Title
plt.title(f"{catchment_name}: Rainfall Pattern Around {event_name}")

# Combine legends from both axes
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper left")

plt.tight_layout()

figure_name = f"{output_prefix}_rainfall_pattern.png"
plt.savefig(figure_name, dpi=300, bbox_inches="tight")

plt.show()

print(f"Saved figure: {figure_name}")



## Step 9 – Interpretation Prompts

Use the outputs above to support written discussion.

Possible points to consider:

### Rainfall intensity
Was the event driven by one high daily rainfall value, or was rainfall spread across several days?

### Antecedent conditions
Was there notable rainfall before the event date? If yes, soils and storage areas may already have been wet, reducing infiltration capacity.

### Catchment response
A sharp rainfall pattern may be associated with a fast catchment response, while prolonged rainfall may indicate saturation-driven runoff.

### NatureInsight® comparison
If NatureInsight® uses a simplified design storm, observed rainfall patterns can help explain differences between modelled and observed hydrographs.

### Wider analysis use
The same workflow can support comparison between catchments, events or rainfall records by applying identical extraction windows and summary metrics.



## Step 10 – Export Event Data

The extracted event rainfall table and summary table are saved for later use in reports, appendices or further analysis.


In [ ]:

# Export event rainfall data and summary table

event_output_csv = f"{output_prefix}_event_window.csv"
summary_output_csv = f"{output_prefix}_summary.csv"

event_rain.to_csv(event_output_csv, index=False)
summary.to_csv(summary_output_csv, index=False)

files.download(event_output_csv)
files.download(summary_output_csv)
files.download(figure_name)

print("Export complete.")
print(f"Saved: {event_output_csv}")
print(f"Saved: {summary_output_csv}")
print(f"Saved: {figure_name}")
